# NSW1 Electricity Price Analysis with Diffusion Model

This notebook implements the independent analysis for the **NSW1** region using a Conditional Diffusion Model. 
It leverages the shared codebase in `src/` for data processing and model definitions.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler

# Add parent directory to path to import src
sys.path.append(os.path.abspath('..'))

from src.data_processing import calculate_caps, apply_capping, create_lagged_features
from src.diffusion_model import DiffusionSchedule, DiffusionDenoiser, ElectricityPriceDataset, sample_with_uncertainty
from src.trainer import train_model
from src.utils import seed_everything

# Set seeds for reproducibility
seed_everything(42)

In [ ]:
# --- Configuration ---
REGION = 'NSW1'
# Load from processed 'Silver' data
DATA_PATH = '../data/prices_wide_clean.parquet'
START_DATE = '2021-10-01'
N_LAGS = 6
PREDICTION_HORIZON = 6
BATCH_SIZE = 256
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Running analysis for {REGION} on {DEVICE}")

## 1. Data Loading & Feature Engineering

We load the pre-processed wide-format data (Silver layer) ensuring data consistency across all state experiments.

In [ ]:
# Load Preprocessed Data
try:
    if DATA_PATH.endswith('.parquet'):
        df_wide = pd.read_parquet(DATA_PATH)
    else:
        df_wide = pd.read_csv(DATA_PATH, parse_dates=['date_time'], index_col='date_time')
    
    # Ensure index is datetime
    if not isinstance(df_wide.index, pd.DatetimeIndex):
         df_wide.index = pd.to_datetime(df_wide.index)
            
    # Filter date range if needed (though preprocess handles start date)
    df_wide = df_wide[df_wide.index >= START_DATE].reset_index()
    
    print(f"Data loaded from {DATA_PATH}. Shape: {df_wide.shape}")
    
except FileNotFoundError:
    print("Processed data file not found. Please run '00_data_preprocess.ipynb' first.")
    # Critical error, stop execution
    raise

In [ ]:
# Calculate and Apply Price Caps (IQR Method)
# Note: We apply caps here as they can be model/experiment specific outlier handling
min_caps, max_caps = calculate_caps(df_wide)
df_capped = apply_capping(df_wide, min_caps, max_caps)

In [ ]:
# Feature Engineering
# Creates lags for ALL regions to predict the target REGION
df_features, feature_cols = create_lagged_features(
    df_capped, 
    target_region=REGION, 
    n_lags=N_LAGS, 
    horizon=PREDICTION_HORIZON
)

print(f"Feature Matrix Shape: {df_features.shape}")
print(f"Target: {REGION} (t+{PREDICTION_HORIZON})")
print(f"Number of features: {len(feature_cols)}")

## 2. Data Preparation for Model

In [ ]:
# Prepare X and y
X = df_features[feature_cols].values
y = df_features['target'].values

# Time-based Train/Test Split (80/20)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Scaling
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

# Create PyTorch Datasets/Loaders
train_dataset = ElectricityPriceDataset(X_train_scaled, y_train_scaled)
test_dataset = ElectricityPriceDataset(X_test_scaled, y_test_scaled)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## 3. model Initialization

In [ ]:
# Diffusion Schedule
schedule = DiffusionSchedule(num_timesteps=1000, device=DEVICE, schedule_type='cosine')

# Model
n_features = X_train_scaled.shape[1]
model = DiffusionDenoiser(
    n_features=n_features, 
    time_emb_dim=64, 
    hidden_dim=256, 
    cfg_dropout=0.1
).to(DEVICE)

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print(model)

## 4. Training

In [ ]:
history = train_model(
    model=model, 
    train_loader=train_loader, 
    val_loader=test_loader, 
    optimizer=optimizer, 
    scheduler=scheduler, 
    device=DEVICE, 
    num_epochs=NUM_EPOCHS, 
    schedule=schedule
)

In [ ]:
# Plot Losses
plt.figure(figsize=(10, 5))
plt.plot(history['train_losses'], label='Train Loss')
plt.plot(history['val_losses'], label='Validation Loss')
plt.title(f'Training History - {REGION}')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(True)
plt.show()

## 5. Save & Evaluation

In [ ]:
# Save Model
save_path = f'../results/models/diffusion_{REGION}.pt'
torch.save(history['best_model_state'], save_path)
print(f"Model saved to {save_path}")

In [ ]:
# Basic Evaluation on Test Set
model.eval()

# Sample a small subset for visualization
n_samples_vis = 100
subset_X = torch.FloatTensor(X_test_scaled[:n_samples_vis]).to(DEVICE)
subset_y_actual = y_test[:n_samples_vis]

with torch.no_grad():
    # Generate 10 samples per input for uncertainty
    predictions_scaled = sample_with_uncertainty(
        model, subset_X, schedule, n_samples=10, cfg_scale=1.5
    )
    
    # Process
    pred_mean_scaled = predictions_scaled.mean(dim=1).cpu().numpy()
    pred_std_scaled = predictions_scaled.std(dim=1).cpu().numpy()
    
    # Inverse transform
    pred_mean = scaler_y.inverse_transform(pred_mean_scaled.reshape(-1, 1)).flatten()
    
# Plot
plt.figure(figsize=(15, 6))
plt.plot(subset_y_actual, label='Actual Price', color='black')
plt.plot(pred_mean, label='Predicted Mean', color='blue')
plt.fill_between(
    range(len(pred_mean)), 
    pred_mean - 2*pred_std_scaled*scaler_y.scale_[0], 
    pred_mean + 2*pred_std_scaled*scaler_y.scale_[0], 
    color='blue', alpha=0.2, label='Uncertainty (2 std)'
)
plt.title(f'{REGION} Price Prediction (First {n_samples_vis} Test Points)')
plt.legend()
plt.show()